# 01 - Benchmark de Avaliação Limpa, Robustez a Ruído e Profiling de Inferência

Reproduz as avaliações de fragmentos curtos (512 bp) que sustentam as **Tabelas 1, 2, 3 e 4** e a **Figura 1** do manuscrito.
- Avalia os 5 modelos de espécie e os 2 modelos de gênero em $\epsilon \in \{0.0, 0.02, 0.05, 0.10, 0.15\}$.
- Perfila eficiência computacional (parâmetros, latência, throughput, GFLOPs).
- Particiona as faixas de F1 da Tabela 4 (, , , ).
- Executa comparação pareada especialista entre ResNet50 e ConvNeXt resolvendo vitórias e empates.


In [ ]:
import os
import sys
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import pandas as pd
import numpy as np

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
)
from viral_cnn.models.registry import (
    REGISTERED_VARIANTS,
    build_registered_model,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.training.profiler import profile_inference_model
from viral_cnn.eval.runner import evaluate_dataloader
from viral_cnn.eval.metrics import (
    compute_species_f1_bands,
    compute_species_specialist_comparison,
)

# 1. Initialize Runtime and Environment
ctx = setup_notebook_runtime()
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")


In [ ]:
# 2. Setup Artifacts and Data Loaders
if ctx.is_smoke_test:
    ensure_smoke_test_artifacts(ctx.data_root)

output_dir = Path("results_reproduction/n02_benchmark")
output_dir.mkdir(parents=True, exist_ok=True)

species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
genus_parquet_dir = Path(ctx.topology.paths["short"]["genus_parquet_dir"])

test_ds_species = ParquetBatchIterableDataset(
    species_parquet_dir,
    batch_size=32 if ctx.is_smoke_test else 1024,
    shuffle=False,
)
test_loader_species = torch.utils.data.DataLoader(test_ds_species, batch_size=None)

test_ds_genus = ParquetBatchIterableDataset(
    genus_parquet_dir,
    batch_size=32 if ctx.is_smoke_test else 1024,
    shuffle=False,
)
test_loader_genus = torch.utils.data.DataLoader(test_ds_genus, batch_size=None)
print("DataLoaders successfully configured.")


In [ ]:
# 3. Model Profiling (Table 2)
profiling_results = []
species_variants = [
    "resnet_clean_species",
    "resnet_noise_species",
    "convnext_zero_species",
    "convnext_transfer_species",
    "convnext_noise_species",
]
genus_variants = [
    "resnet_genus",
    "convnext_genus",
]

for name in species_variants + genus_variants:
    model = build_registered_model(name, device=ctx.device)
    prof = profile_inference_model(
        model,
        input_size=(32, 512),
        device=ctx.device,
        num_warmup=2 if ctx.is_smoke_test else 5,
        num_runs=5 if ctx.is_smoke_test else 20,
    )
    prof["model_name"] = name
    profiling_results.append(prof)
    print(f"Profiled {name}: {prof['total_params']:,} params, {prof['avg_latency_ms']:.2f} ms")

df_prof = pd.DataFrame(profiling_results)
df_prof.to_csv(output_dir / "table2_profiling.csv", index=False)
print("Profiling table saved.")


In [ ]:
# 4. Clean Evaluation and Controlled Noise Robustness (Table 1, Table 3, Fig 1)
noise_levels = [0.0, 0.02, 0.05, 0.10, 0.15]
max_batches = 2 if ctx.is_smoke_test else None

all_evaluations = {}
summary_rows = []

for name in species_variants:
    print(f"Evaluating Species Model: {name}")
    model = build_registered_model(name, device=ctx.device)
    num_classes = 5 if ctx.is_smoke_test else 13683
    evals = evaluate_dataloader(
        model,
        test_loader_species,
        device=ctx.device,
        noise_rates=noise_levels,
        seed=42,
        model_name=name,
        num_classes=num_classes,
        max_batches=max_batches,
    )
    all_evaluations[name] = evals
    for eps, res in evals.items():
        summary_rows.append({
            "model_name": name,
            "rank": "species",
            "nominal_noise": eps,
            "effective_noise": res.effective_noise_rate,
            "top1_accuracy": res.metrics.top1_accuracy,
            "top5_accuracy": res.metrics.top5_accuracy,
            "macro_f1": res.metrics.macro_f1,
            "weighted_f1": res.metrics.weighted_f1,
            "loss": res.metrics.cross_entropy_loss,
            "n_samples": res.metrics.n_samples,
        })

for name in genus_variants:
    print(f"Evaluating Genus Model: {name}")
    model = build_registered_model(name, device=ctx.device)
    num_classes = 3 if ctx.is_smoke_test else 2702
    evals = evaluate_dataloader(
        model,
        test_loader_genus,
        device=ctx.device,
        noise_rates=noise_levels,
        seed=42,
        model_name=name,
        num_classes=num_classes,
        max_batches=max_batches,
    )
    all_evaluations[name] = evals
    for eps, res in evals.items():
        summary_rows.append({
            "model_name": name,
            "rank": "genus",
            "nominal_noise": eps,
            "effective_noise": res.effective_noise_rate,
            "top1_accuracy": res.metrics.top1_accuracy,
            "top5_accuracy": res.metrics.top5_accuracy,
            "macro_f1": res.metrics.macro_f1,
            "weighted_f1": res.metrics.weighted_f1,
            "loss": res.metrics.cross_entropy_loss,
            "n_samples": res.metrics.n_samples,
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(output_dir / "benchmark_clean_noise_summary.csv", index=False)
print("Benchmark summary saved.")


In [ ]:
# 5. Table 4 F1 Performance Bands at epsilon = 0.05
f1_bands_rows = []
for name in species_variants:
    res_05 = all_evaluations[name][0.05]
    for b_key, b_info in res_05.f1_bands["bands"].items():
        f1_bands_rows.append({
            "model_name": name,
            "band_key": b_key,
            "interval": b_info["range"],
            "class_count": b_info["count"],
            "percent": b_info["percent"],
            "mean_confidence": b_info.get("mean_confidence", 0.0),
        })

df_f1_bands = pd.DataFrame(f1_bands_rows)
df_f1_bands.to_csv(output_dir / "table4_f1_bands.csv", index=False)
print("Table 4 F1 bands saved.")


In [ ]:
# 6. Specialist Comparison: ResNet50 vs ConvNeXt (Table 4 / Note 45)
res_resnet = all_evaluations["resnet_noise_species"][0.05]
res_convnext = all_evaluations["convnext_noise_species"][0.05]

y_true = [p.truth_label for p in res_resnet.predictions]
preds_res = [p.predicted_label for p in res_resnet.predictions]
preds_conv = [p.predicted_label for p in res_convnext.predictions]
confs_res = [p.confidence for p in res_resnet.predictions]
confs_conv = [p.confidence for p in res_convnext.predictions]

comp_res = compute_species_specialist_comparison(
    y_true=y_true,
    preds_resnet=preds_res,
    preds_convnext=preds_conv,
    confs_resnet=confs_res,
    confs_convnext=confs_conv,
    num_classes=5 if ctx.is_smoke_test else 13683,
)

print(f"ResNet Superior: {comp_res['resnet_superior']}")
print(f"ConvNeXt Superior: {comp_res['convnext_superior']}")
print(f"Ties: {comp_res['ties']}")
print(f"Both Zero: {comp_res['both_zero']}")
print(f"Total Evaluated Species: {comp_res['total_evaluated_species']}")

df_specialist = pd.DataFrame(comp_res["per_species"])
df_specialist.to_csv(output_dir / "specialist_comparison.csv", index=False)
print("Specialist comparison saved.")


In [ ]:
# 7. Write Execution Manifest
manifest = write_notebook_manifest(
    notebook_id="01_benchmark_clean_and_noise",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    metrics_summary={
        "evaluated_models": len(species_variants) + len(genus_variants),
        "noise_levels": noise_levels,
    },
    output_json_path=output_dir / "manifest.json",
)
print("Manifest successfully written.")
